In [ ]:
import pandas as pd
import numpy as np

# 데이터 로드
df = pd.read_csv(r'C:\workspace\finalproject\data\google trend\trends_originall_us.csv')
df['date'] = pd.to_datetime(df['date'])

# axis 선택 (트렌드 관련)
target_axis = ['트렌드', 'K뷰티_수요', '효능_FitScore']
df_filtered = df[df['axis'].isin(target_axis)]

# 기간 설정
recent = df_filtered[df_filtered['date'] >= '2025-04-01']
prev = df_filtered[(df_filtered['date'] >= '2024-04-01') & (df_filtered['date'] < '2025-04-01')]

# 키워드별 최근 1년 vs 전년 평균
recent_keyword = recent.groupby(['category', 'keyword'])['value'].mean()
prev_keyword = prev.groupby(['category', 'keyword'])['value'].mean()

# 공통 키워드만
common_idx = recent_keyword.index.intersection(prev_keyword.index)
recent_keyword = recent_keyword[common_idx]
prev_keyword = prev_keyword[common_idx]

# 상승/하락 판단
is_rising = (recent_keyword > prev_keyword).reset_index()
is_rising.columns = ['category', 'keyword', 'is_rising']

# 카테고리별 상승 키워드 비율 → 트렌드 점수
trend_score = is_rising.groupby('category')['is_rising'].mean() * 100
trend_score = trend_score.round(2)

print("=== 카테고리별 트렌드 점수 (0~100점) ===")
print(trend_score)
print()

# 카테고리 매핑 (소분류 → 대분류)
category_mapping = {
    'skincare_Face': '스킨케어',
    'skincare_lipcare': '스킨케어',
    'skincare_eyecare': '스킨케어',
    'makeup_Eyes': '메이크업',
    'makeup_Face': '메이크업',
    'makeup_Lips': '메이크업',
    'makeup_Makeup Palettes': '메이크업',
    'suncare': '썬케어',
    'maskpack': '마스크팩'
}

print("=== 소분류별 트렌드 점수 매핑 ===")
for sub, main in category_mapping.items():
    score = trend_score.get(main, 0)
    print(f"{sub}: {score}점 ({main} 트렌드)")

=== 카테고리별 트렌드 점수 (0~100점) ===
category
마스크팩     60.0
메이크업     60.0
스킨케어     40.0
썬케어     100.0
Name: is_rising, dtype: float64

=== 소분류별 트렌드 점수 매핑 ===
skincare_Face: 40.0점 (스킨케어 트렌드)
skincare_lipcare: 40.0점 (스킨케어 트렌드)
skincare_eyecare: 40.0점 (스킨케어 트렌드)
makeup_Eyes: 60.0점 (메이크업 트렌드)
makeup_Face: 60.0점 (메이크업 트렌드)
makeup_Lips: 60.0점 (메이크업 트렌드)
makeup_Makeup Palettes: 60.0점 (메이크업 트렌드)
suncare: 100.0점 (썬케어 트렌드)
maskpack: 60.0점 (마스크팩 트렌드)


In [8]:
# 썬케어 키워드별 상승/하락 확인
suncare_detail = is_rising[is_rising['category'] == '썬케어']
print("=== 썬케어 키워드별 상승/하락 ===")
print(suncare_detail[['keyword', 'is_rising']].to_string())
print()
print(f"총 키워드 수: {len(suncare_detail)}")
print(f"상승 키워드 수: {suncare_detail['is_rising'].sum()}")

=== 썬케어 키워드별 상승/하락 ===
                       keyword  is_rising
29  beauty of joseon sunscreen       True
30             clear sunscreen       True
31           elta md sunscreen       True
32         invisible sunscreen       True
33                k-beauty SPF       True
34            korean sun serum       True
35            korean sunscreen       True
36    la roche posay sunscreen       True
37          skin1004 sunscreen       True
38           sun bum sunscreen       True

총 키워드 수: 10
상승 키워드 수: 10


In [9]:
# 썬케어 전체 데이터 샘플
suncare_data = df_filtered[df_filtered['category'] == '썬케어']
print(f"썬케어 총 행 수: {len(suncare_data)}")
print()
print("=== axis 분포 ===")
print(suncare_data['axis'].value_counts())
print()
print("=== 키워드 목록 ===")
print(suncare_data['keyword'].unique())
print()
print("=== 날짜별 샘플 ===")
print(suncare_data[suncare_data['keyword'] == 'korean sunscreen'].head(10).to_string())

썬케어 총 행 수: 2620

=== axis 분포 ===
axis
K뷰티_수요    1310
트렌드       1310
Name: count, dtype: int64

=== 키워드 목록 ===
<ArrowStringArray>
['beauty of joseon sunscreen',               'k-beauty SPF',
           'korean sun serum',           'korean sunscreen',
         'skin1004 sunscreen',            'clear sunscreen',
          'elta md sunscreen',        'invisible sunscreen',
   'la roche posay sunscreen',          'sun bum sunscreen']
Length: 10, dtype: str

=== 날짜별 샘플 ===
            date category    axis           keyword  value
55020 2021-04-11      썬케어  K뷰티_수요  korean sunscreen     17
55021 2021-04-18      썬케어  K뷰티_수요  korean sunscreen     14
55022 2021-04-25      썬케어  K뷰티_수요  korean sunscreen     16
55023 2021-05-02      썬케어  K뷰티_수요  korean sunscreen     10
55024 2021-05-09      썬케어  K뷰티_수요  korean sunscreen     10
55025 2021-05-16      썬케어  K뷰티_수요  korean sunscreen     11
55026 2021-05-23      썬케어  K뷰티_수요  korean sunscreen     12
55027 2021-05-30      썬케어  K뷰티_수요  korean sunscreen    

In [10]:
df.value.describe()

count    75718.000000
mean        10.946921
std         15.684063
min          0.000000
25%          1.000000
50%          4.000000
75%         15.000000
max        100.000000
Name: value, dtype: float64

In [11]:
# 카테고리별 axis 분포 확인
print(df.groupby(['category', 'axis']).size().reset_index(name='count').to_string())

   category         axis  count
0      마스크팩       K뷰티_수요   1310
1      마스크팩           가격   2358
2      마스크팩      그린티_바이럴   1310
3      마스크팩        나이/타겟   1310
4      마스크팩        나이_타겟   1048
5      마스크팩      브랜드카테고리   1310
6      마스크팩           성분   1310
7      마스크팩           제형   1310
8      마스크팩          트렌드   1310
9      마스크팩         피부타입   1310
10     마스크팩      핵심수요/효능   1310
11     마스크팩  효능_FitScore   1310
12     메이크업       K뷰티_수요   1048
13     메이크업           가격   1310
14     메이크업           나이   1048
15     메이크업        나이_타겟   1048
16     메이크업           색상   1310
17     메이크업        색상/호수   1310
18     메이크업           성분   1310
19     메이크업           제형   2096
20     메이크업         카테고리   1572
21     메이크업          트렌드   1310
22     메이크업         피부타입   1310
23     메이크업           호수   1310
24     메이크업           효능   1310
25     스킨케어       K뷰티_수요   1310
26     스킨케어           가격   2620
27     스킨케어           나이   1310
28     스킨케어        나이_타겟   1048
29     스킨케어           성분   2620
30     스

In [13]:
# 트렌드 axis 키워드별 상세 확인
trend_only = df[df['axis'] == '트렌드']

for cat in ['마스크팩', '메이크업', '스킨케어', '썬케어']:
    print(f"\n=== {cat} 트렌드 키워드 ===")
    cat_data = trend_only[trend_only['category'] == cat]
    print(f"키워드 수: {cat_data['keyword'].nunique()}")
    print(cat_data['keyword'].unique())


=== 마스크팩 트렌드 키워드 ===
키워드 수: 5
<ArrowStringArray>
['biodance collagen mask',  'green tea mask review',        'keana rice mask',
       'laneige lip mask',            'zombie pack']
Length: 5, dtype: str

=== 메이크업 트렌드 키워드 ===
키워드 수: 5
<ArrowStringArray>
[  'clean girl makeup',             'lip oil',        'merit makeup',
   'minimalist makeup', 'rare beauty lip oil']
Length: 5, dtype: str

=== 스킨케어 트렌드 키워드 ===
키워드 수: 5
<ArrowStringArray>
[       'bubble skincare',            'glow recipe', 'hailey bieber skincare',
        'preppy skincare',         'rhode skincare']
Length: 5, dtype: str

=== 썬케어 트렌드 키워드 ===
키워드 수: 5
<ArrowStringArray>
[         'clear sunscreen',        'elta md sunscreen',
      'invisible sunscreen', 'la roche posay sunscreen',
        'sun bum sunscreen']
Length: 5, dtype: str


In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\workspace\finalproject\data\google trend\trends_originall_us.csv')
df['date'] = pd.to_datetime(df['date'])

# axis 필터링 없이 전체 키워드 사용
target_axis = ['트렌드', 'K뷰티_수요', '효능_FitScore', '카테고리', '핵심수요/효능']
recent = df[df['date'] >= '2025-04-01']
prev = df[(df['date'] >= '2024-04-01') & (df['date'] < '2025-04-01')]

# 키워드별 평균
recent_keyword = recent.groupby(['category', 'keyword'])['value'].mean()
prev_keyword = prev.groupby(['category', 'keyword'])['value'].mean()

# 공통 키워드만
common_idx = recent_keyword.index.intersection(prev_keyword.index)
recent_keyword = recent_keyword[common_idx]
prev_keyword = prev_keyword[common_idx]

# 상승/하락 판단
is_rising = (recent_keyword > prev_keyword).reset_index()
is_rising.columns = ['category', 'keyword', 'is_rising']

# 카테고리별 키워드 수 확인
print("=== 카테고리별 키워드 수 ===")
print(is_rising.groupby('category')['keyword'].count())
print()

# 카테고리별 상승 비율 → 트렌드 점수
trend_score = is_rising.groupby('category')['is_rising'].mean() * 100
trend_score = trend_score.round(2)

print("=== 카테고리별 트렌드 점수 (0~100점) ===")
print(trend_score)

=== 카테고리별 키워드 수 ===
category
마스크팩    47
메이크업    54
스킨케어    57
썬케어     56
Name: keyword, dtype: int64

=== 카테고리별 트렌드 점수 (0~100점) ===
category
마스크팩    78.72
메이크업    92.59
스킨케어    92.98
썬케어     96.43
Name: is_rising, dtype: float64


In [18]:
# 카테고리별 axis 분포 확인
print(is_rising.merge(df[['category','keyword','axis']].drop_duplicates(), 
                       on=['category','keyword'])['axis'].value_counts())

axis
제형             26
성분             25
피부타입           22
가격             20
트렌드            20
K뷰티_수요         19
나이_타겟          16
효능_FitScore    15
카테고리           13
효능             12
핵심수요/효능        10
나이/타겟          10
브랜드카테고리        10
나이              9
그린티_바이럴         5
색상              5
호수              5
색상/호수           5
성분2             5
SPF/PA          5
SPF/호수          5
무백탁/어리          5
호수/톤            5
Name: count, dtype: int64


In [19]:
# 성분 관련 axis 키워드 확인
ingredient_axis = df[df['axis'].isin(['성분', '성분2'])]
print("=== 성분 axis 카테고리별 키워드 ===")
for cat in ['마스크팩', '메이크업', '스킨케어', '썬케어']:
    cat_data = ingredient_axis[ingredient_axis['category'] == cat]
    print(f"\n{cat}:")
    print(cat_data['keyword'].unique())

=== 성분 axis 카테고리별 키워드 ===

마스크팩:
<ArrowStringArray>
[       'charcoal mask',        'collagen mask', 'hyaluronic acid mask',
     'niacinamide mask',         'retinol mask']
Length: 5, dtype: str

메이크업:
<ArrowStringArray>
[  'clean makeup', 'mineral makeup', 'natural makeup', 'spf foundation',
   'vegan makeup']
Length: 5, dtype: str

스킨케어:
<ArrowStringArray>
[          'aha bha',          'centella',          'ceramide',
   'hyaluronic acid',       'niacinamide',  'peptide skincare',
           'retinol',       'snail mucin',   'tranexamic acid',
   'vitamin c serum',      'azelaic acid', 'centella asiatica',
           'peptide']
Length: 13, dtype: str

썬케어:
<ArrowStringArray>
[        'chemical sunscreen',          'mineral sunscreen',
      'niacinamide sunscreen', 'titanium dioxide sunscreen',
       'zinc oxide sunscreen']
Length: 5, dtype: str
